# 02. Data Cleaning & Preprocessing — CampusWell RiskLab
> **Student Mental Health Risk Analysis and Prediction**
> *Strict Data Leakage Prevention, Empirical Imputation Benchmarking, Outlier Analysis, and Evidence-Driven Feature Engineering*

---

### Educational and Synthetic Data Disclaimer
This dataset is **entirely synthetic**, created for machine learning pedagogy, portfolio demonstration, and educational defense. It does not contain clinical diagnoses or psychiatric records and must never be utilized for real-world diagnostic, administrative, or disciplinary decisions.

### Methodological Objectives of this Notebook
1. **Data Leakage Prevention:** Establish a frozen 80/20 stratified split (`X_train`, `X_test`, random_state=42) before any transformation, scaling, or imputation.
2. **Missing-Value Strategy Experimentation:** Conduct 5-fold cross-validated artificial masking exclusively on `X_train` to empirically compare Mean, Median, KNN, and Iterative imputation using Mean Absolute Error (MAE).
3. **Evidence-Based Imputation Selection:** Select the optimal imputation strategy per column based on reconstruction accuracy, fold stability, and model simplicity.
4. **Outlier Investigation & Treatment:** Analyze IQR fences and test whether robust scaling or clipping is justified.
5. **Controlled Feature Engineering Experiments:** Formulate and benchmark candidate interaction features (`sleep_to_screen_ratio`, `academic_workload_ratio`, `non_social_screen_time`, `stress_support_ratio`) using identical CV folds on training data.
6. **Feature Selection & Ablation:** Exclude arbitrary identifiers (`Student_ID`) and discard non-performing interaction terms.
7. **Unified Scikit-Learn Pipeline:** Construct a leak-free `ColumnTransformer` and pipeline supporting out-of-sample inference and Joblib serialization.

## Section 1: Environment Setup & Configuration
We import necessary statistical, data manipulation, and scikit-learn modules. Zero comments or docstrings are used in code cells, adhering strictly to the project coding standard.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from src.config import (
    DATA_PATH,
    TARGET_COLUMN,
    STUDENT_ID_COLUMN,
    NUMERICAL_FEATURES,
    ENGINEERED_NUMERICAL_FEATURES,
    ORDINAL_FEATURES,
    NOMINAL_FEATURES,
    BINARY_FEATURES,
    TARGET_CLASSES,
    RANDOM_STATE,
    TEST_SIZE,
    DATA_PROCESSED_DIR,
    TRAIN_FEATURES_PATH,
    TRAIN_TARGET_PATH,
    TEST_FEATURES_PATH,
    TEST_TARGET_PATH,
)
from src.data.load_data import load_dataset
from src.features.preprocessing import (
    FeatureEngineer,
    build_preprocessor,
    build_model_pipeline,
)
print("Libraries and configuration successfully imported.")


Libraries and configuration successfully imported.


## Section 2: Data Ingestion & Identifier Exclusion
We load the raw CSV dataset and immediately exclude `Student_ID`. Identifiers carry arbitrary tracking indexes that possess zero generalized predictive power; retaining them would risk spurious correlation or model memorization.

In [2]:
df = load_dataset(DATA_PATH)
if STUDENT_ID_COLUMN in df.columns:
    df = df.drop(columns=[STUDENT_ID_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

pd.DataFrame({
    "Attribute": ["Total Observations", "Predictive Features", "Target Variable"],
    "Value": [len(df), X.shape[1], y.name]
})

,Attribute,Value
0,Total Observations,2000
1,Predictive Features,15
2,Target Variable,Mental_Health_Risk


## Section 3: Stratified Train/Test Partitioning
To guarantee zero data leakage, we partition the dataset into 80% training ($N=1,600$) and 20% independent testing ($N=400$) using stratified sampling on `Mental_Health_Risk`. The test partition remains completely untouched until final model evaluation.